# 97 — Multi-Template Weighted Delta-ML

**Motivation:** nb76 delta-ML picks the *single* nearest training template for each test compound. This is fragile when the top-1 neighbor is noisy or the compound lies between two structural clusters.

**Strategy:**
1. For each query compound, find ALL training neighbors within Tanimoto window [0.35, 0.90] (up to K=10)
2. For each neighbor, predict delta(pEC50) using a fold-trained delta LGBM
3. Combine: `pred = sum_k( w_k * (pEC50_k + delta_k) ) / sum_k(w_k)` where `w_k = sim_k^2`
4. Fall back to direct LGBM when no neighbors found in window

**Expected gain:** More robust than single template; reduces variance from noisy top-1 neighbor.

In [1]:
import os, sys, warnings
os.environ["PYTHONIOENCODING"] = "utf-8"
if hasattr(sys.stdout, "reconfigure"): sys.stdout.reconfigure(encoding="utf-8")
sys.path.insert(0, "../src")
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import lightgbm as lgb
from scipy import stats
from pathlib import Path
from pxr.data import load_train, load_test
from pxr.featurize import combined, impute
from pxr.eval import rae, scaffold_kfold_indices
from pxr.chem import bemis_murcko, morgan_fp_batch, standardize_smiles, compute_physchem
from pxr.paths import DATA_PROCESSED, DATA_EXTERNAL, SUBMISSIONS
SEED = 42; N_FOLDS = 5
LGBM = dict(n_estimators=1000, num_leaves=64, learning_rate=0.05,
            min_child_samples=10, subsample=0.8, colsample_bytree=0.8,
            reg_alpha=0.1, reg_lambda=0.1, random_state=SEED, verbose=-1, n_jobs=4)


In [2]:
def full_metrics(y_true, y_pred, cp=None, label=""):
    yt = np.asarray(y_true, float); yp = np.asarray(y_pred, float)
    msk = np.isfinite(yt) & np.isfinite(yp); yt, yp = yt[msk], yp[msk]
    mae = float(np.mean(np.abs(yt-yp)))
    rae_v = mae / float(np.mean(np.abs(yt-yt.mean()))) if yt.std()>0 else float("nan")
    r2  = 1-np.sum((yt-yp)**2)/np.sum((yt-yt.mean())**2) if yt.std()>0 else float("nan")
    pr, _ = stats.pearsonr(yt, yp); sp, _ = stats.spearmanr(yt, yp)
    kt, _ = stats.kendalltau(yt, yp)
    m = dict(RAE=rae_v, MAE=mae, R2=float(r2), Pearson=float(pr),
             Spearman=float(sp), Kendall=float(kt))
    if cp is not None and hasattr(cp, "iterrows") and len(cp) > 0:
        c=t=0
        for _,row in cp.iterrows():
            ia,ii = int(row.get("idx_active",-1)), int(row.get("idx_inactive",-1))
            if 0<=ia<len(yp) and 0<=ii<len(yp): c+=int(yp[ia]>yp[ii]); t+=1
        m["Cliff_acc"] = c/t if t else float("nan")
    if label:
        ca = f"  Cliff={m.get('Cliff_acc',float('nan')):.3f}" if "Cliff_acc" in m else ""
        print(f"  [{label}] RAE={rae_v:.4f} MAE={mae:.4f} R2={r2:.4f} "
              f"r={pr:.4f} rho={sp:.4f} tau={kt:.4f}{ca}")
    return m


In [3]:
tr = load_train(); te = load_test()
y_tr = tr["pec50"].values.astype(np.float64)
scaffolds = tr["smiles"].map(bemis_murcko).tolist()
splits = scaffold_kfold_indices(scaffolds, N_FOLDS, SEED)
active_mask = y_tr >= 5.5
X_tr = impute(combined(tr["smiles"].tolist()))
X_te = impute(combined(te["smiles"].tolist()))
fps_tr = morgan_fp_batch(tr["smiles"].tolist()).astype(np.float32)
fps_te = morgan_fp_batch(te["smiles"].tolist()).astype(np.float32)
cliff_pairs = (pd.read_parquet(DATA_PROCESSED/"cliff_pairs.parquet")
               if (DATA_PROCESSED/"cliff_pairs.parquet").exists() else pd.DataFrame())
if len(cliff_pairs) > 0:
    s2i = {s:i for i,s in enumerate(tr["smiles"].tolist())}
    ac = "cliff_active_smiles" if "cliff_active_smiles" in cliff_pairs.columns else "smiles_a"
    ic = "cliff_inactive_smiles" if "cliff_inactive_smiles" in cliff_pairs.columns else "smiles_b"
    cliff_pairs["idx_active"]   = cliff_pairs[ac].map(s2i)
    cliff_pairs["idx_inactive"] = cliff_pairs[ic].map(s2i)
    cliff_pairs = cliff_pairs.dropna(subset=["idx_active","idx_inactive"])
    cliff_pairs[["idx_active","idx_inactive"]] = cliff_pairs[["idx_active","idx_inactive"]].astype(int)
print(f"Train {len(tr):,}  Test {len(te):,}  Cliffs {len(cliff_pairs)}")


Train 4,139  Test 513  Cliffs 0


In [4]:
# --- Build delta-pair training dataset ---
SIM_LO = 0.35   # minimum similarity to be a useful template
SIM_HI = 0.90   # upper cap: very similar pairs have trivially near-zero delta
MAX_PAIRS = 400_000
props = ["mw","logp","tpsa","hbd","hba","rotbonds","rings"]

print("Computing physchem descriptors...", flush=True)
phys_tr = tr["smiles"].map(compute_physchem).tolist()
phys_arr = np.array([[p.get(k,0) or 0 for k in props] for p in phys_tr], dtype=np.float32)

print("Computing pairwise Tanimoto (train x train)...", flush=True)
dot_tt = (fps_tr @ fps_tr.T).astype(np.float32)
rowsum = fps_tr.sum(1).astype(np.float32)
union_tt = rowsum[:,None] + rowsum[None,:] - dot_tt
tanimoto_tr = np.where(union_tt>0, dot_tt/union_tt, 0.0)
np.fill_diagonal(tanimoto_tr, 0.0)

i_idx, j_idx = np.where((tanimoto_tr >= SIM_LO) & (tanimoto_tr <= SIM_HI))
mask_upper = i_idx < j_idx
i_idx, j_idx = i_idx[mask_upper], j_idx[mask_upper]
print(f"Pairs in sim window [{SIM_LO},{SIM_HI}]: {len(i_idx):,}")

rng = np.random.default_rng(SEED)
if len(i_idx) > MAX_PAIRS:
    sel = rng.choice(len(i_idx), MAX_PAIRS, replace=False)
    i_idx, j_idx = i_idx[sel], j_idx[sel]
    print(f"Downsampled to {MAX_PAIRS:,} pairs")


Computing physchem descriptors...


Computing pairwise Tanimoto (train x train)...


Pairs in sim window [0.35,0.9]: 5,177


In [5]:
# --- Build compressed delta features ---
# We compress morgan (2048-d) to 64-d averaged blocks to keep feature dim manageable

def compress_fp(fp, out_dim=64):
    """Average-pool fp from (N, D) to (N, out_dim)."""
    N, D = fp.shape
    block = D // out_dim
    return fp[:, :block*out_dim].reshape(N, out_dim, block).mean(-1).astype(np.float32)

def make_delta_feats(fp_anchor, fp_query, sim_col, anchor_pec50, phys_diff):
    """Features for delta prediction: common/diff substructure + physchem delta."""
    fp_common = np.minimum(fp_anchor, fp_query).astype(np.float32)
    fp_diff   = np.abs(fp_anchor - fp_query).astype(np.float32)
    c64 = compress_fp(fp_common)
    d64 = compress_fp(fp_diff)
    return np.hstack([c64, d64, sim_col, anchor_pec50[:,None], phys_diff])

fps_i = fps_tr[i_idx]; fps_j = fps_tr[j_idx]
sims_ij = tanimoto_tr[i_idx, j_idx][:,None]
phys_diff_ij = phys_arr[j_idx] - phys_arr[i_idx]
phys_diff_ji = -phys_diff_ij
y_delta_ij = y_tr[j_idx] - y_tr[i_idx]

print("Building feature matrices...", flush=True)
F_ij = make_delta_feats(fps_i, fps_j, sims_ij, y_tr[i_idx], phys_diff_ij)
F_ji = make_delta_feats(fps_j, fps_i, sims_ij, y_tr[j_idx], phys_diff_ji)
F_all = np.vstack([F_ij, F_ji])
y_all = np.concatenate([y_delta_ij, -y_delta_ij])
print(f"Delta dataset: {F_all.shape}  delta range [{y_all.min():.2f}, {y_all.max():.2f}]")


Building feature matrices...


Delta dataset: (10354, 137)  delta range [-4.68, 4.68]


In [6]:
# --- Train global delta model on all pairs ---
print("Training global delta LGBM on all pairs...", flush=True)
DELTA_LGBM = dict(n_estimators=600, num_leaves=63, learning_rate=0.05,
                  min_child_samples=20, subsample=0.8, colsample_bytree=0.7,
                  reg_alpha=0.05, reg_lambda=0.1, random_state=SEED, verbose=-1, n_jobs=4)
delta_model = lgb.LGBMRegressor(**DELTA_LGBM)
delta_model.fit(F_all, y_all, callbacks=[lgb.log_evaluation(-1)])
print("Delta model trained.", flush=True)


Training global delta LGBM on all pairs...


Delta model trained.


In [7]:
# --- Multi-template prediction function ---
K_NEIGHBORS = 10   # max templates to use

def multi_template_predict(fps_query, fps_ref, y_ref, phys_query, phys_ref,
                           sim_matrix, delta_model, direct_preds,
                           sim_lo=SIM_LO, sim_hi=SIM_HI, k=K_NEIGHBORS):
    """
    For each query, use up to k templates in [sim_lo, sim_hi] window.
    Returns weighted prediction (falls back to direct if no templates found).
    """
    N = len(fps_query)
    preds = np.full(N, np.nan)
    n_templates = np.zeros(N, dtype=int)

    for qi in range(N):
        # Find neighbors in similarity window
        sim_row = sim_matrix[qi]
        cand_mask = (sim_row >= sim_lo) & (sim_row <= sim_hi)
        cand_idx = np.where(cand_mask)[0]
        if len(cand_idx) == 0:
            # Fall back to direct prediction
            preds[qi] = direct_preds[qi]
            continue
        # Take top-k by similarity
        cand_sims = sim_row[cand_idx]
        top_k_order = np.argsort(-cand_sims)[:k]
        cand_idx = cand_idx[top_k_order]
        cand_sims = cand_sims[top_k_order]
        n_templates[qi] = len(cand_idx)

        # Build features for all templates at once
        fp_q_rep = np.tile(fps_query[qi:qi+1], (len(cand_idx), 1))
        fp_refs  = fps_ref[cand_idx]
        sims_col = cand_sims[:,None]
        anc_pec50 = y_ref[cand_idx]
        phys_d = phys_query[qi:qi+1] - phys_ref[cand_idx]
        F_k = make_delta_feats(fp_refs, fp_q_rep, sims_col, anc_pec50, phys_d)
        delta_k = delta_model.predict(F_k)
        template_preds = y_ref[cand_idx] + delta_k
        # Weight by sim^2
        weights = cand_sims ** 2
        preds[qi] = np.average(template_preds, weights=weights)

    return preds, n_templates

print(f"Multi-template function ready (K={K_NEIGHBORS}, window=[{SIM_LO},{SIM_HI}])")


Multi-template function ready (K=10, window=[0.35,0.9])


In [8]:
# --- Scaffold 5-fold CV ---
print("\n=== Scaffold 5-fold CV ===", flush=True)
oof_multi_delta = np.full(len(y_tr), np.nan)
oof_direct      = np.full(len(y_tr), np.nan)

for fold, (tr_idx, va_idx) in enumerate(splits):
    # Direct LGBM
    m_dir = lgb.train(LGBM, lgb.Dataset(X_tr[tr_idx], label=y_tr[tr_idx]),
                      valid_sets=[lgb.Dataset(X_tr[va_idx], label=y_tr[va_idx])],
                      callbacks=[lgb.early_stopping(50,verbose=False), lgb.log_evaluation(-1)])
    oof_direct[va_idx] = m_dir.predict(X_tr[va_idx])

    # Build sim matrix: val x fold-train
    fps_val  = fps_tr[va_idx]
    fps_ftrain = fps_tr[tr_idx]
    dot_vf = (fps_val @ fps_ftrain.T).astype(np.float32)
    rs_v = fps_val.sum(1)[:,None]; rs_f = fps_ftrain.sum(1)[None,:]
    sim_vf = dot_vf / np.maximum(rs_v + rs_f - dot_vf, 1e-6)

    phys_val = phys_arr[va_idx]
    phys_ftrain = phys_arr[tr_idx]

    preds_mt, n_tmpl = multi_template_predict(
        fps_val, fps_ftrain, y_tr[tr_idx], phys_val, phys_ftrain,
        sim_vf, delta_model, oof_direct[va_idx]
    )
    oof_multi_delta[va_idx] = preds_mt

    r_dir = rae(y_tr[va_idx], oof_direct[va_idx])
    r_mt  = rae(y_tr[va_idx], oof_multi_delta[va_idx])
    avg_tmpl = float(n_tmpl.mean())
    print(f"  fold {fold+1}  direct={r_dir:.4f}  multi_delta={r_mt:.4f}  "
          f"avg_templates={avg_tmpl:.1f}", flush=True)

m_dir = full_metrics(y_tr, oof_direct,      cliff_pairs, "direct_lgbm")
m_mt  = full_metrics(y_tr, oof_multi_delta, cliff_pairs, "multi_template_delta")



=== Scaffold 5-fold CV ===


  fold 1  direct=0.4982  multi_delta=0.2912  avg_templates=1.9


  fold 2  direct=0.5759  multi_delta=0.3193  avg_templates=1.8


  fold 3  direct=0.6021  multi_delta=0.3612  avg_templates=1.7


  fold 4  direct=0.5665  multi_delta=0.3294  avg_templates=1.7


  fold 5  direct=0.6033  multi_delta=0.3461  avg_templates=1.7


  [direct_lgbm] RAE=0.5643 MAE=0.5134 R2=0.5991 r=0.7740 rho=0.7268 tau=0.5345
  [multi_template_delta] RAE=0.3266 MAE=0.2972 R2=0.8178 r=0.9060 rho=0.8754 tau=0.7237


In [9]:
# --- Sweep blend alpha: multi_delta vs direct ---
best_alpha, best_rae_v = 0.0, full_metrics(y_tr, oof_direct)["RAE"]
for alpha in np.arange(0.0, 1.05, 0.1):
    blended = alpha*oof_multi_delta + (1-alpha)*oof_direct
    r = rae(y_tr[np.isfinite(blended)], blended[np.isfinite(blended)])
    print(f"  alpha={alpha:.1f}  RAE={r:.4f}")
    if r < best_rae_v:
        best_rae_v, best_alpha = r, alpha

oof = best_alpha*oof_multi_delta + (1-best_alpha)*oof_direct
m_blend = full_metrics(y_tr, oof, cliff_pairs, f"blend(a={best_alpha:.1f})")
print(f"\nBest blend alpha={best_alpha:.1f}  OOF RAE={best_rae_v:.4f}")
print("\n" + pd.DataFrame([m_dir, m_mt, m_blend],
                           index=["direct","multi_delta",f"blend_{best_alpha:.1f}"]).round(4).to_string())


  alpha=0.0  RAE=0.5643
  alpha=0.1  RAE=0.5381
  alpha=0.2  RAE=0.5120
  alpha=0.3  RAE=0.4863
  alpha=0.4  RAE=0.4608
  alpha=0.5  RAE=0.4357
  alpha=0.6  RAE=0.4113
  alpha=0.7  RAE=0.3878
  alpha=0.8  RAE=0.3654
  alpha=0.9  RAE=0.3446
  alpha=1.0  RAE=0.3266
  [blend(a=1.0)] RAE=0.3266 MAE=0.2972 R2=0.8178 r=0.9060 rho=0.8754 tau=0.7237

Best blend alpha=1.0  OOF RAE=0.3266

                RAE     MAE      R2  Pearson  Spearman  Kendall
direct       0.5643  0.5134  0.5991    0.774    0.7268   0.5345
multi_delta  0.3266  0.2972  0.8178    0.906    0.8754   0.7237
blend_1.0    0.3266  0.2972  0.8178    0.906    0.8754   0.7237


In [10]:
# --- Final test predictions ---
print("\nFitting final direct model on all train...", flush=True)
m_final = lgb.train(LGBM, lgb.Dataset(X_tr, label=y_tr), callbacks=[lgb.log_evaluation(-1)])
te_direct = m_final.predict(X_te)

# Test-train sim matrix for multi-template
print("Computing test-train similarity...", flush=True)
dot_tet = (fps_te @ fps_tr.T).astype(np.float32)
rs_te = fps_te.sum(1)[:,None]; rs_tr_v = fps_tr.sum(1)[None,:]
sim_te_tr = dot_tet / np.maximum(rs_te + rs_tr_v - dot_tet, 1e-6)

phys_te = np.array([[p.get(k,0) or 0 for k in props]
                     for p in te["smiles"].map(compute_physchem)], dtype=np.float32)

print("Running multi-template on test...", flush=True)
te_mt, n_tmpl_te = multi_template_predict(
    fps_te, fps_tr, y_tr, phys_te, phys_arr,
    sim_te_tr, delta_model, te_direct
)
print(f"Test: avg templates used = {n_tmpl_te.mean():.1f}  "
      f"min={n_tmpl_te.min()}  max={n_tmpl_te.max()}")

te_preds = best_alpha*te_mt + (1-best_alpha)*te_direct
te_preds = np.clip(te_preds, y_tr.min()-0.5, y_tr.max()+0.5)

np.save(DATA_PROCESSED/"oof_multi_template_delta.npy", oof)
np.save(DATA_PROCESSED/"te_oof_multi_template_delta.npy", te_preds)
sub = pd.DataFrame({"Molecule Name": te["name"].values, "pEC50": te_preds})
assert len(sub)==513 and sub["pEC50"].notna().all()
p = SUBMISSIONS/"97_multi_template_delta.csv"; sub.to_csv(p, index=False)
print(f"Saved {p}")
print(f"Test: min={te_preds.min():.2f} med={np.median(te_preds):.2f} max={te_preds.max():.2f}")
print(f"\n*** nb97 OOF RAE = {m_blend['RAE']:.4f} ***")



Fitting final direct model on all train...


Computing test-train similarity...


Running multi-template on test...


Test: avg templates used = 3.1  min=0  max=10
Saved D:\Users\ashenoy00000\.windsurf\OpenADMET-pxr-challenge\submissions\97_multi_template_delta.csv
Test: min=2.80 med=4.72 max=5.74

*** nb97 OOF RAE = 0.3266 ***
